# 555 多變量 LSTM：獨立每日執行版

本版只執行新版資料處理、模型、風控回測、指標視覺化與工作日誌，不讀取另一版 Notebook 的任何檔案，不做新舊合併或比較。前端之後再由各自的標準化輸出讀取結果。

**每日資料流**：獨立下載五檔及 ^TWII 價格 → 合併上游真實新聞情緒 → 重算技術指標與大盤風險 → 時序切分訓練 → 回測與指標 → 檢討以前保存的下一日建議 → 另訓練部署模型 → 保存新建議與本次快照。

**執行方式**：安裝套件，準備 `lstm_multivariate_features_2026.csv` 後，依序執行。價格預設由 yfinance 獨立下載；離線時可指定本版自己保存的 `historical_prices.csv` 或同格式的外部價格檔。不需要舊版 ZIP。

**同步設計**：使用 Asia/Taipei、相同資料截止日定義、150 日測試／60 日驗證、10 日視窗、費率、指標定義、固定股數參考組合方向與共同輸出欄位。`evaluation_protocol.json` 和資料雜湊供未來接收端驗收；本版不會修改或檢查另一份程式，另一份仍須採相同規格才能公平比較。

**驗證邊界**：程式以合成資料驗證流程與核心計算；目前沒有真實新聞 CSV 及 TensorFlow 訓練結果，未驗證真實預測績效。輸出均待執行後產生。

**資料責任**：本版消費上游新聞情緒檔，不會捏造情緒。新聞爬蟲、FinBERT-Chinese、逐篇發布時間稽核及 GitHub 排程仍需另接資料端／repo。新聞稽核尚未確認时，報告及建議會明確標示。

## 0．套件與輸入
Colab 或 Cursor/Jupyter 皆可執行。每日上游情緒檔至少含 `Date,ticker,Daily_Sentiment`；有新聞發布時間時使用 `Latest_News_Timestamp`。價格由本版直接取得，獨立運作。

In [ ]:
%pip install -q tensorflow yfinance pandas numpy matplotlib scikit-learn joblib exchange-calendars

## 1．獨立參數與同步規格
`AS_OF_DATE=None`：台北時間 15:00 後才將當日視為可用，並以台股交易日曆選最近已結束交易日。資料若未更新到該日會停止，不偷偷改用較早日期。可指定 `AS_OF_DATE="2026-10-13"` 重現歷史截止日。

兩份獨立程式需共同採用這份日期截止與研究規格。手續費／稅率為研究參數；股息不再投資，現金收益為零，允許零碎股數。回測採理論 close-to-close 執行假設；完整收盤資料不一定可於同一收盤價成交，報告須揭露，實務化時兩版一起改成交時點與標籤。

In [ ]:
import os
os.environ.setdefault('TF_DETERMINISTIC_OPS', '1')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import json, hashlib, platform
import sklearn
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib
try:
    import tensorflow as tf
except ImportError:
    tf = None
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x.to_string() if hasattr(x, 'to_string') else x)

TICKERS = ['2330.TW', '2317.TW', '2603.TW', '2002.TW', '2891.TW']
MARKET_TICKER = '^TWII'
FEATURES = ['Close', 'MA20', 'RSI_14', 'MACD_Hist', 'Market_Vol_20d', 'Daily_Sentiment']
FEATURE_CSV = Path(os.environ.get('LSTM_FEATURE_CSV', 'lstm_multivariate_features_2026.csv'))
LOCAL_PRICE_CSV = os.environ.get('LSTM_PRICE_CSV') or None
START_DATE = '2014-01-01'
AS_OF_DATE = os.environ.get('LSTM_AS_OF_DATE') or None
DATA_READY_TIME = '15:00'
AUTO_ADJUST = False
OUTPUT_ROOT = Path(os.environ.get('LSTM_OUTPUT_DIR', 'lstm_multivariate_outputs'))
PROTOCOL_VERSION = 'lstm-evaluation-v1'
STRATEGY_ID = '555_multivariate_dynamic'
RUN_SCENARIO_FORECAST = False
SCENARIO_SENTIMENT = 0.0  # 只用于明示假設的未來情境，不是新聞實測值。
SCENARIO_DAYS = 30
TIME_STEP = 10
TEST_DAYS = 150
VALIDATION_DAYS = 60
MIN_TRAIN_SAMPLES = 80
EPOCHS = 50
PATIENCE = 5
BATCH_SIZE = 32
SEED = 42
INITIAL_CAPITAL = 100000.0
BROKER_FEE_RATE = 0.001425
SELL_TAX_RATE = 0.003
LIQUIDATE_AT_END = True
ANNUAL_TRADING_DAYS = 252
RISK_FREE_RATE = 0.0
DIRECTION_ZERO_TOL_PCT = 1e-8
SENTIMENT_BULL = 0.3
SENTIMENT_BEAR = -0.3
MAX_WEIGHT_LIMIT = 0.30
RISK_EXPOSURE = 0.50
SAVE_MODELS = os.environ.get("LSTM_SAVE_MODELS", "1") == "1"
RUN_DEPLOYMENT_REFIT = True  # 另訓練全部已知歷史的模型，只用於下一交易日建議。
RUN_ABLATIONS = False       # 可開啟策略規則消融；不等於移除 LSTM 的情緒輸入。
NEWS_TIME_AUDIT_CONFIRMED = False  # 只有確認上游發布時間映射後才可改為 True。
EXTRA_CLOSED_DATES = []

CREATED_AT = datetime.now(ZoneInfo('Asia/Taipei'))
RUN_ID = CREATED_AT.strftime('%Y%m%dT%H%M%S%f')
OUTPUT_DIR = OUTPUT_ROOT / 'runs' / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
MODELS, PREDICTIONS, SPLIT_AUDIT, HISTORIES = {}, {}, [], {}
DEPLOYMENT_MODELS = {}
READINESS_NOTES = []
INPUT_HASHES = {}
FIGURE_INDEX = []
_original_show = plt.show
def show_and_save_figures(*args, **kwargs):
    folder = OUTPUT_DIR / 'figures'
    folder.mkdir(parents=True, exist_ok=True)
    for number in plt.get_fignums():
        figure = plt.figure(number)
        name = f'{len(FIGURE_INDEX)+1:02d}.png'
        path = folder / name
        figure.savefig(path, dpi=150, bbox_inches='tight')
        title = figure.axes[0].get_title() if figure.axes else ''
        FIGURE_INDEX.append({'File': str(path.relative_to(OUTPUT_DIR)), 'Title': title})
    _original_show(*args, **kwargs)
    plt.close('all')
plt.show = show_and_save_figures

## 2．独立價格來源與真實情緒
五檔與大盤依日期交集對齊，使用本版參數設定的原始收盤價。缺少截止日價格會明確失敗；保存來源、缺值日與 SHA-256。
MA20、RSI、MACD 與年化 20 日波動由同一價格重新計算。`Target_Next_Return` 若由上游提供，會核對下一交易日百分比報酬；不會以 CSV 內的未來報酬作為特徵。
新聞缺值不自動填 0；只有上游確認无新聞時才可填中性 0。發布時間晚於訊號日 13:30 者拒收；無時區時間解讀為 Asia/Taipei。周末、假日新聞應由上游先映射至正確交易日。

In [ ]:
def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def clean_dates(frame):
    frame = frame.copy()
    def trading_date(value):
        stamp = pd.Timestamp(value)
        if pd.isna(stamp):
            return pd.NaT
        if stamp.tzinfo is not None:
            stamp = stamp.tz_convert('Asia/Taipei').tz_localize(None)
        return stamp.normalize()
    frame['Date'] = pd.to_datetime(frame['Date'].map(trading_date), errors='raise')
    if frame['Date'].isna().any():
        raise ValueError('Date 含缺值。')
    return frame

def technical_features(close):
    result = pd.DataFrame({'Close': close.astype(float)})
    result['MA20'] = close.rolling(20).mean()
    delta = close.diff()
    gain = delta.clip(lower=0).ewm(alpha=1/14, adjust=False, min_periods=14).mean()
    loss = (-delta.clip(upper=0)).ewm(alpha=1/14, adjust=False, min_periods=14).mean()
    result['RSI_14'] = 100 - 100/(1 + gain/loss.replace(0, np.nan))
    result.loc[(loss == 0) & (gain > 0), 'RSI_14'] = 100.0
    result.loc[(loss == 0) & (gain == 0), 'RSI_14'] = 50.0
    macd = (close.ewm(span=12, adjust=False, min_periods=12).mean()
            - close.ewm(span=26, adjust=False, min_periods=26).mean())
    result['MACD_Hist'] = macd - macd.ewm(span=9, adjust=False, min_periods=9).mean()
    return result

def build_market_risk(market):
    risk = pd.DataFrame({'Market_Close': market})
    risk['Market_Return'] = market.pct_change(fill_method=None)
    risk['Market_Vol_20d'] = risk.Market_Return.rolling(20).std(ddof=1)*np.sqrt(252)*100
    previous = risk.Market_Vol_20d.shift(1)
    risk['Risk_Threshold_Pct'] = (previous.expanding(min_periods=60).mean()
                                + previous.expanding(min_periods=60).std(ddof=1))
    risk['Risk_Off'] = risk.Market_Vol_20d > risk.Risk_Threshold_Pct
    return risk

def read_csv_checked(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'缺少 {path}。請準備真實情緒 CSV 或指定本地價格檔。')
    INPUT_HASHES[str(path)] = sha256_file(path)
    return pd.read_csv(path)

def trading_sessions(start, end):
    import exchange_calendars as xcals
    start, end = pd.Timestamp(start).normalize(), pd.Timestamp(end).normalize()
    cal = xcals.get_calendar('XTAI', start=start-pd.Timedelta(days=7), end=end+pd.Timedelta(days=7))
    sessions = pd.DatetimeIndex(cal.sessions_in_range(start, end))
    if sessions.tz is not None:
        sessions = sessions.tz_localize(None)
    return sessions.normalize().difference(pd.DatetimeIndex(pd.to_datetime(EXTRA_CLOSED_DATES)))

def resolve_as_of_date(now=None):
    now = CREATED_AT if now is None else pd.Timestamp(now)
    if now.tzinfo is None:
        now = now.tz_localize('Asia/Taipei')
    now = now.astimezone(ZoneInfo('Asia/Taipei'))
    today = pd.Timestamp(now.date())
    hour, minute = map(int, DATA_READY_TIME.split(':'))
    upper = today if (now.hour, now.minute) >= (hour, minute) else today-pd.Timedelta(days=1)
    sessions = trading_sessions(upper-pd.Timedelta(days=60), upper)
    if len(sessions) == 0:
        raise ValueError('無法解析最新完整交易日。')
    latest = sessions[-1]
    chosen = latest if AS_OF_DATE is None else pd.Timestamp(AS_OF_DATE).normalize()
    if chosen > latest or chosen not in trading_sessions(chosen-pd.Timedelta(days=7), chosen):
        raise ValueError('AS_OF_DATE 尚未完成交易、不是交易日或晚於可用資料日。')
    if chosen < pd.Timestamp(START_DATE):
        raise ValueError('AS_OF_DATE 早於 START_DATE。')
    return chosen, latest

RESOLVED_AS_OF, LATEST_COMPLETED_SESSION = resolve_as_of_date()
RUN_MODE = 'daily' if RESOLVED_AS_OF == LATEST_COMPLETED_SESSION else 'historical_replay'
END_DATE_EXCLUSIVE = RESOLVED_AS_OF+pd.Timedelta(days=1)

def download_or_read_prices():
    required = TICKERS+[MARKET_TICKER]
    if LOCAL_PRICE_CSV:
        raw = read_csv_checked(LOCAL_PRICE_CSV)
        source_name = 'local_csv'
    else:
        try:
            import yfinance as yf
        except ImportError as exc:
            raise ImportError('請安裝 yfinance，或設定 LOCAL_PRICE_CSV。') from exc
        parts = []
        for ticker in required:
            frame = yf.download(ticker, start=START_DATE,
                                end=END_DATE_EXCLUSIVE.strftime('%Y-%m-%d'),
                                auto_adjust=AUTO_ADJUST, progress=False, threads=False, timeout=30)
            if frame is None or frame.empty:
                raise ValueError(f'{ticker} 價格下載失敗，請重試；不使用假資料。')
            if isinstance(frame.columns, pd.MultiIndex):
                frame.columns = frame.columns.get_level_values(0)
            frame = frame.reset_index()
            if 'Date' not in frame:
                raise ValueError(f'{ticker} 下載資料缺 Date。')
            frame['ticker'] = ticker
            parts.append(frame[['Date', 'ticker', 'Close']])
        raw = pd.concat(parts, ignore_index=True)
        source_name = 'yfinance:auto_adjust=False' if not AUTO_ADJUST else 'yfinance:auto_adjust=True'
    if not {'Date', 'ticker', 'Close'}.issubset(raw):
        raise ValueError('價格輸入必須含 Date,ticker,Close，且包含五檔及 ^TWII。')
    raw = clean_dates(raw)
    raw['ticker'] = raw.ticker.astype(str)
    raw['Close'] = pd.to_numeric(raw.Close, errors='raise')
    raw = raw.loc[raw.ticker.isin(required) & (raw.Date >= pd.Timestamp(START_DATE))
                  & (raw.Date <= RESOLVED_AS_OF)].copy()
    if raw.duplicated(['Date', 'ticker']).any():
        raise ValueError('價格 Date/ticker 重複。')
    if not set(required).issubset(set(raw.ticker)):
        raise ValueError('價格缺少五檔股票或 ^TWII。')
    panel = raw.pivot(index='Date', columns='ticker', values='Close')[required].sort_index()
    if not np.isfinite(panel.to_numpy()[~np.isnan(panel.to_numpy())]).all() or (panel <= 0).any().any():
        raise ValueError('價格包含非有限或非正值。')
    absent = panel.index[panel.isna().any(axis=1)]
    audit = []
    for ticker in required:
        part = raw.loc[raw.ticker == ticker].dropna(subset=['Close']).sort_values('Date')
        if part.empty or part.Date.iloc[-1] != RESOLVED_AS_OF:
            raise ValueError(f'{ticker} 未更新到 {RESOLVED_AS_OF.date()}，已停止；不變更本次截止日。')
        audit.append({'Ticker': ticker, 'Original_Rows': len(part),
                      'First_Date': part.Date.iloc[0], 'Last_Date': part.Date.iloc[-1],
                      'Source': source_name})
    panel = panel.dropna()
    # 最近評估與暖機期內不允許資料缺口，避免把非相鄰交易日當作下一日。
    minimum = TEST_DAYS+VALIDATION_DAYS+MIN_TRAIN_SAMPLES+TIME_STEP+90
    expected = trading_sessions(panel.index[0], RESOLVED_AS_OF)
    expected_tail = expected[-minimum:]
    missing_tail = expected_tail.difference(panel.index)
    unexpected = panel.index.difference(expected)
    # Yahoo price snapshot is shared with 444; keep its actual observation dates.
    # XTAI calendar and Yahoo can disagree on a few historical sessions.
    # Never synthesize prices, shift target dates, or silently drop shared dates.
    # scripts/prices.py already validates the latest 400 sessions of the shared
    # snapshot before either model runs. Keep this discrepancy visible in output.
    if len(missing_tail) or len(unexpected):
        calendar_audit = pd.concat([
            pd.DataFrame({'Date': missing_tail, 'Issue': 'calendar_session_without_shared_price'}),
            pd.DataFrame({'Date': unexpected, 'Issue': 'shared_price_date_not_in_XTAI_calendar'}),
        ], ignore_index=True)
        calendar_audit.to_csv(OUTPUT_DIR/'price_calendar_discrepancies.csv', index=False,
                              encoding='utf-8-sig')
        print('[555][CALENDAR WARNING] 共用股價與 XTAI 日曆日期不完全相符；'
              f'缺日={missing_tail.strftime("%Y-%m-%d").tolist()}，'
              f'額外價格日={unexpected.strftime("%Y-%m-%d").tolist()}。'
              '保留共用價格日期供 444/555 相同樣本比較，不補造價格。', flush=True)
        READINESS_NOTES.append('XTAI 日曆與共用股價日期有差異；詳見 price_calendar_discrepancies.csv。')
    for row in audit:
        row['Common_Rows'] = len(panel)
    raw.to_csv(OUTPUT_DIR/'historical_prices.csv', index=False, encoding='utf-8-sig')
    INPUT_HASHES['price_snapshot'] = sha256_file(OUTPUT_DIR/'historical_prices.csv')
    pd.DataFrame(audit).to_csv(OUTPUT_DIR/'data_audit.csv', index=False, encoding='utf-8-sig')
    pd.DataFrame({'Removed_Incomplete_Date': absent}).to_csv(OUTPUT_DIR/'removed_price_dates.csv', index=False)
    return panel

def prepare_data():
    panel = download_or_read_prices()
    raw = clean_dates(read_csv_checked(FEATURE_CSV))
    if not {'Date', 'ticker', 'Daily_Sentiment'}.issubset(raw):
        raise ValueError('情緒檔至少需 Date,ticker,Daily_Sentiment。')
    raw['ticker'] = raw.ticker.astype(str)
    raw = raw.loc[raw.ticker.isin(TICKERS) & (raw.Date <= RESOLVED_AS_OF)].copy()
    if raw.duplicated(['Date', 'ticker']).any():
        raise ValueError('情緒 Date/ticker 重複。')
    raw['Daily_Sentiment'] = pd.to_numeric(raw.Daily_Sentiment, errors='raise')
    values = raw.Daily_Sentiment.dropna()
    if not np.isfinite(values).all() or not values.between(-1, 1).all():
        raise ValueError('情緒必須是 [-1,1] 的有限數值。')
    if 'Latest_News_Timestamp' in raw:
        valid = raw.Latest_News_Timestamp.notna()
        def taipei_timestamp(value):
            stamp = pd.Timestamp(value)
            return (stamp.tz_localize('Asia/Taipei') if stamp.tzinfo is None else stamp).tz_convert('UTC')
        published = raw.loc[valid, 'Latest_News_Timestamp'].map(taipei_timestamp)
        cutoff = ((raw.loc[valid, 'Date']+pd.Timedelta(hours=13, minutes=30))
                  .dt.tz_localize('Asia/Taipei').dt.tz_convert('UTC'))
        if (published > cutoff).any():
            raise ValueError('包含訊號日 13:30 後的新聞；請修正上游日期映射。')
    if not NEWS_TIME_AUDIT_CONFIRMED:
        READINESS_NOTES.append('新聞逐篇發布時間、來源及交易日映射尚待上游稽核。')
    raw.to_csv(OUTPUT_DIR/'sentiment_snapshot.csv', index=False, encoding='utf-8-sig')
    risk = build_market_risk(panel[MARKET_TICKER])
    risk.index.name = 'Date'
    risk.to_csv(OUTPUT_DIR/'market_risk_features.csv', encoding='utf-8-sig')
    frames = {}
    next_dates = pd.Series(panel.index, index=panel.index).shift(-1)
    for ticker in TICKERS:
        supplied = raw.loc[raw.ticker == ticker].set_index('Date').sort_index()
        common = panel.index.intersection(supplied.index)
        if 'Close' in supplied:
            observed = pd.to_numeric(supplied.loc[common, 'Close'], errors='raise')
            if not np.isfinite(observed).all() or not np.allclose(observed, panel.loc[common,ticker], rtol=1e-6, atol=1e-4):
                raise ValueError(f'{ticker} 情緒檔 Close 與獨立價格不一致，請核對來源／復權。')
        f = technical_features(panel[ticker])
        f['Daily_Sentiment'] = supplied.Daily_Sentiment.reindex(panel.index)
        f['Market_Vol_20d'] = risk.Market_Vol_20d
        f['Risk_Threshold_Pct'] = risk.Risk_Threshold_Pct
        f['Risk_Off'] = risk.Risk_Off
        f['Target_Date'] = next_dates
        f['Target_Next_Return'] = panel[ticker].pct_change(fill_method=None).shift(-1)*100
        if 'Target_Next_Return' in supplied:
            y = pd.to_numeric(supplied.Target_Next_Return, errors='raise').reindex(panel.index)
            mask = y.notna() & f.Target_Next_Return.notna()
            if not np.isfinite(y[mask]).all() or not np.allclose(y[mask], f.loc[mask,'Target_Next_Return'], rtol=1e-4, atol=1e-4):
                raise ValueError(f'{ticker} 上游 Target_Next_Return 的單位或標籤日期不符。')
        if not np.isfinite(f[FEATURES].iloc[-TIME_STEP:].to_numpy()).all():
            raise ValueError(f'{ticker} 最新十日特徵不足，不能完成當日工作或輸出過期建議。')
        frames[ticker] = f
    feature_export = pd.concat([f.reset_index().assign(ticker=t) for t,f in frames.items()], ignore_index=True)
    feature_export.to_csv(OUTPUT_DIR/'model_feature_snapshot.csv', index=False, encoding='utf-8-sig')
    return panel, frames, risk

PRICE_PANEL, FEATURE_FRAMES, MARKET_RISK = prepare_data()
if len(PRICE_PANEL) < TEST_DAYS+VALIDATION_DAYS+TIME_STEP+MIN_TRAIN_SAMPLES:
    raise ValueError('有效歷史長度不足。')
TARGET_DATES = PRICE_PANEL.index[-TEST_DAYS:]
VALIDATION_TARGET_DATES = PRICE_PANEL.index[-TEST_DAYS-VALIDATION_DAYS:-TEST_DAYS]
print(f'本次模式：{RUN_MODE}；資料截至 {RESOLVED_AS_OF.date()}')
print(f'測試報酬實現日：{TARGET_DATES[0].date()} ～ {TARGET_DATES[-1].date()}')

## 3．統一標籤日期切分與訓練
最近 150 個共同交易日是測試目標日，之前 60 個是固定驗證目標日；不是每檔各取最後 60 個有資料的日子。
訓練資料早於驗證起點，Scaler 只 fit 訓練視窗。新聞覆蓋導致的實際訓練範圍明確匯出。所有個股需相同有效訓練及驗證日期。

In [ ]:
def direction(values):
    a = np.asarray(values, dtype=float)
    return np.where(np.abs(a) <= DIRECTION_ZERO_TOL_PCT, 0, np.sign(a)).astype(int)

def make_windows(frame, target_dates=TARGET_DATES):
    values = frame[FEATURES].to_numpy(dtype=float)
    candidate = []
    for end in range(TIME_STEP-1, len(frame)-1):
        start = end-TIME_STEP+1
        if (np.isfinite(values[start:end+1]).all()
                and np.isfinite(frame.Target_Next_Return.iloc[end])):
            candidate.append(end)
    ends = np.asarray(candidate, dtype=int)
    if len(ends) == 0:
        raise ValueError('沒有有效連續視窗。請確認情緒覆蓋、技術指標及風險暖機期。')
    label_dates = pd.DatetimeIndex(frame.Target_Date.iloc[ends])
    test_mask = label_dates.isin(target_dates)
    if not label_dates[test_mask].equals(pd.DatetimeIndex(target_dates)):
        missing = pd.DatetimeIndex(target_dates).difference(label_dates)
        raise ValueError(f'固定測試期有缺特徵或視窗：{missing.strftime("%Y-%m-%d").tolist()}。不能自行縮短測試期。')
    val_indices = np.where(label_dates.isin(VALIDATION_TARGET_DATES))[0]
    if not label_dates[val_indices].equals(VALIDATION_TARGET_DATES):
        raise ValueError('固定 60 日驗證期有缺特徵，請補資料；不移動驗證日期。')
    train_indices = np.where(label_dates < VALIDATION_TARGET_DATES[0])[0]
    if len(train_indices) < MIN_TRAIN_SAMPLES:
        raise ValueError('有效訓練資料不足。請補新聞歷史，不可拿驗證或測試資料訓練。')
    test_ends = ends[test_mask]
    if frame.Risk_Threshold_Pct.iloc[test_ends].isna().any():
        raise ValueError('測試期風控門檻缺值。')
    train_rows = np.unique(np.concatenate([np.arange(ends[k]-TIME_STEP+1, ends[k]+1)
                                           for k in train_indices]))
    scaler = MinMaxScaler().fit(values[train_rows])
    scaled = np.full_like(values, np.nan)
    valid_rows = np.isfinite(values).all(axis=1)
    scaled[valid_rows] = scaler.transform(values[valid_rows])
    X = np.stack([scaled[end-TIME_STEP+1:end+1] for end in ends])
    y = frame.Target_Next_Return.iloc[ends].to_numpy(dtype=float)
    return {'X_train': X[train_indices], 'y_train': y[train_indices],
            'X_val': X[val_indices], 'y_val': y[val_indices],
            'X_test': X[test_mask], 'y_test': y[test_mask], 'scaler': scaler,
            'X_all': X, 'y_all': y, 'ends': ends,
            'test_ends': ends[test_mask], 'train_rows': train_rows,
            'train_dates': label_dates[train_indices], 'val_dates': label_dates[val_indices],
            'test_dates': label_dates[test_mask], 'all_dates': label_dates}

def model_factory():
    if tf is None:
        raise ImportError('請先安裝 TensorFlow；目前沒有執行真實訓練。')
    tf.keras.utils.set_random_seed(SEED)
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(TIME_STEP, len(FEATURES))),
        tf.keras.layers.LSTM(50, return_sequences=True), tf.keras.layers.Dropout(.2),
        tf.keras.layers.LSTM(50), tf.keras.layers.Dropout(.2), tf.keras.layers.Dense(1)])
    model.compile(optimizer='adam', loss='mse')
    return model

def save_bundle(bundle, ticker, purpose):
    if SAVE_MODELS:
        folder = OUTPUT_DIR / 'models'
        folder.mkdir(exist_ok=True)
        name = ticker.replace('.', '_') + '_' + purpose
        bundle['model'].save(folder / (name + '.keras'))
        joblib.dump(bundle['scaler'], folder / (name + '_scaler.joblib'))
        (folder / (name + '_metadata.json')).write_text(json.dumps({
            'ticker': ticker, 'features': FEATURES, 'time_step': TIME_STEP,
            'target': 'next-trading-day return, percent', 'purpose': purpose,
            'seed': SEED, 'best_epoch': bundle['best_epoch'],
            'run_id': RUN_ID, 'as_of_date': RESOLVED_AS_OF.date().isoformat(),
            'fit_scope': 'pre-test train, validation-selected' if purpose == 'backtest' else 'all known labeled history'}, ensure_ascii=False, indent=2), encoding='utf-8')

def train_all_models():
    windows = {t: make_windows(FEATURE_FRAMES[t]) for t in TICKERS}
    # 所有個股用相同訓練及驗證標籤日，避免新聞覆蓋不同造成暗中變動。
    train_dates = windows[TICKERS[0]]['train_dates']
    val_dates = windows[TICKERS[0]]['val_dates']
    for ticker, data in windows.items():
        if not data['train_dates'].equals(train_dates) or not data['val_dates'].equals(val_dates):
            raise ValueError(f'{ticker} 的訓練/驗證有效日期與其他股票不同。請先補齊或共同選定有效歷史期。')
    for ticker, data in windows.items():
        model = model_factory()
        callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=PATIENCE,
                                                    restore_best_weights=True)
        print(ticker, 'train/val/test =', len(data['X_train']), len(data['X_val']), len(data['X_test']))
        history = model.fit(data['X_train'], data['y_train'], epochs=EPOCHS,
                            batch_size=BATCH_SIZE, shuffle=False,
                            validation_data=(data['X_val'], data['y_val']),
                            callbacks=[callback], verbose=0)
        losses = np.asarray(history.history['val_loss'])
        if not np.isfinite(losses).all():
            raise ValueError(f'{ticker} 的訓練損失非有限值。')
        predicted = model.predict(data['X_test'], verbose=0).ravel()
        if not np.isfinite(predicted).all() or (predicted <= -100).any():
            raise ValueError(f'{ticker} 產生無效或低於 -100% 的預測。')
        f = FEATURE_FRAMES[ticker].iloc[data['test_ends']]
        prediction = pd.DataFrame({
            'Date': data['test_dates'], 'Signal_Date': f.index, 'Ticker': ticker,
            'Previous_Close': f.Close.to_numpy(),
            'Actual_Return_Pct': data['y_test'], 'Predicted_Return_Pct': predicted,
            'Daily_Sentiment': f.Daily_Sentiment.to_numpy(),
            'Market_Vol_20d': f.Market_Vol_20d.to_numpy(),
            'Risk_Threshold_Pct': f.Risk_Threshold_Pct.to_numpy(),
            'Risk_Off': f.Risk_Off.to_numpy(dtype=bool)})
        prediction['Actual_Close'] = prediction.Previous_Close*(1+prediction.Actual_Return_Pct/100)
        prediction['Predicted_Close'] = prediction.Previous_Close*(1+predicted/100)
        prediction['Actual_Direction'] = direction(prediction.Actual_Return_Pct)
        prediction['Predicted_Direction'] = direction(predicted)
        prediction['Direction_Correct'] = prediction.Actual_Direction.eq(prediction.Predicted_Direction)
        prediction['Rolling_Accuracy_Pct'] = prediction.Direction_Correct.rolling(20).mean()*100
        audit = {'Ticker': ticker, 'Train_Start': str(data['train_dates'][0].date()),
                 'Train_End': str(data['train_dates'][-1].date()),
                 'Validation_Start': str(data['val_dates'][0].date()),
                 'Validation_End': str(data['val_dates'][-1].date()),
                 'Test_Start': str(data['test_dates'][0].date()),
                 'Test_End': str(data['test_dates'][-1].date()),
                 'Train_Samples': len(data['X_train']), 'Validation_Samples': len(data['X_val']),
                 'Test_Samples': len(data['X_test']), 'Best_Epoch': int(losses.argmin()+1)}
        bundle = {'model': model, 'scaler': data['scaler'], 'data': data,
                  'best_epoch': audit['Best_Epoch']}
        MODELS[ticker] = bundle
        PREDICTIONS[ticker] = prediction
        SPLIT_AUDIT.append(audit)
        HISTORIES[ticker] = history.history
        save_bundle(bundle, ticker, 'backtest')
    return pd.DataFrame(SPLIT_AUDIT)

split_audit = train_all_models()
display(split_audit)

## 4．動態持股及自洽的交易成本
依模型方向與情緒配置 0%、20%、30%，超過曝險上限才按比例縮小。剩餘資金為現金（現金收益假設為 0）。高波動最高持股 50%；不是必定持股 50%。
每次調倉依「調倉前持股市值」與「調倉後目標市值」計算買賣，交易成本由資金內支付；不將權重差直接當成交易金額。單股 30% 上限是調倉時的目標上限，日內價格漂移不保證仍低於 30%。

In [ ]:
def allocate(predictions, sentiments, risk_off, use_sentiment=True, use_risk=True):
    pred = np.asarray(predictions, dtype=float)
    sent = np.asarray(sentiments, dtype=float)
    if not np.isfinite(pred).all() or not np.isfinite(sent).all():
        raise ValueError('配置輸入不得含 NaN/Inf。')
    raw = np.full(len(pred), .2)
    if use_sentiment:
        raw[(pred > 0) & (sent > SENTIMENT_BULL)] = MAX_WEIGHT_LIMIT
        raw[(pred < 0) | (sent < SENTIMENT_BEAR)] = 0.0
    else:
        raw[pred < 0] = 0.0
    exposure = RISK_EXPOSURE if (use_risk and risk_off) else 1.0
    if raw.sum() > exposure:
        raw *= exposure/raw.sum()
    if (raw < 0).any() or raw.sum() > exposure+1e-10 or raw.max() > MAX_WEIGHT_LIMIT+1e-10:
        raise ValueError('配置超出風控上限。')
    return raw

def rebalance(current_values, cash, target_weights, fee=BROKER_FEE_RATE, tax=SELL_TAX_RATE):
    current = np.asarray(current_values, dtype=float)
    weights = np.asarray(target_weights, dtype=float)
    total = float(current.sum()+cash)
    if total <= 0 or cash < -1e-7 or (current < 0).any() or (weights < 0).any() or weights.sum() > 1+1e-10:
        raise ValueError('調倉資金或權重無效。')
    if not (0 <= fee < 1 and 0 <= tax < 1 and fee+tax < 1):
        raise ValueError('成本參數無效。')
    # 解 post_nav + buy_fee + sell_fee + sell_tax = pre_nav。
    low, high = 0.0, total
    for _ in range(70):
        mid = (low+high)/2
        delta = weights*mid-current
        costs = delta.clip(min=0).sum()*fee + (-delta.clip(max=0)).sum()*(fee+tax)
        if mid+costs > total:
            high = mid
        else:
            low = mid
    after = (low+high)/2
    target = weights*after
    delta = target-current
    buys = float(delta.clip(min=0).sum())
    sells = float((-delta.clip(max=0)).sum())
    ledger = {'Buy_Notional': buys, 'Sell_Notional': sells, 'Buy_Fee': buys*fee,
              'Sell_Fee': sells*fee, 'Sell_Tax': sells*tax,
              'Total_Cost': buys*fee+sells*(fee+tax),
              'Turnover': (buys+sells)/total}
    remaining = float(after-target.sum())
    if not np.isclose(target.sum()+remaining+ledger['Total_Cost'], total, atol=1e-6):
        raise AssertionError('調倉資金未守恆。')
    return target, remaining, ledger

def run_backtest(use_sentiment=True, use_risk=True):
    first = PREDICTIONS[TICKERS[0]]
    first_signal = pd.Timestamp(first.Signal_Date.iloc[0])
    shares = np.zeros(len(TICKERS))
    cash = INITIAL_CAPITAL
    nav_rows = [{'Date': first_signal, 'NAV': INITIAL_CAPITAL, 'Cash': INITIAL_CAPITAL}]
    weights_rows, costs_rows, direction_rows = [], [], []
    for i, target_date in enumerate(TARGET_DATES):
        rows = [PREDICTIONS[t].iloc[i] for t in TICKERS]
        signal_date = pd.Timestamp(rows[0].Signal_Date)
        if any(pd.Timestamp(r.Date) != target_date or pd.Timestamp(r.Signal_Date) != signal_date for r in rows):
            raise ValueError('五檔預測日期不一致。')
        price_before = PRICE_PANEL.loc[signal_date, TICKERS].to_numpy(dtype=float)
        price_after = PRICE_PANEL.loc[target_date, TICKERS].to_numpy(dtype=float)
        old_values = shares*price_before
        pre_nav = float(old_values.sum()+cash)
        weights = allocate([r.Predicted_Return_Pct for r in rows], [r.Daily_Sentiment for r in rows],
                           bool(rows[0].Risk_Off), use_sentiment, use_risk)
        target_values, cash, ledger = rebalance(old_values, cash, weights)
        shares = target_values/price_before
        ending = float((shares*price_after).sum()+cash)
        # 固定股數參考組合用於模型方向評估；和動態策略方向分開。
        reference_shares = INITIAL_CAPITAL*.2/PRICE_PANEL.loc[first_signal, TICKERS].to_numpy(dtype=float)
        reference_weights = reference_shares*price_before
        reference_weights /= reference_weights.sum()
        actual_returns = np.array([r.Actual_Return_Pct for r in rows])
        predicted_returns = np.array([r.Predicted_Return_Pct for r in rows])
        reference_actual = float(reference_weights@actual_returns)
        reference_pred = float(reference_weights@predicted_returns)
        dynamic_actual = float(weights@actual_returns)
        dynamic_pred = float(weights@predicted_returns)
        cost_ratio = ledger['Total_Cost']/pre_nav
        next_expected_net = (1-cost_ratio)*(1+dynamic_pred/100)-1
        for j, ticker in enumerate(TICKERS):
            weights_rows.append({'Date': target_date, 'Signal_Date': signal_date, 'Ticker': ticker,
                                 'Weight': weights[j], 'Cash_Weight': 1-weights.sum(),
                                 'Predicted_Return_Pct': predicted_returns[j],
                                 'Daily_Sentiment': rows[j].Daily_Sentiment,
                                 'Risk_Off': bool(rows[0].Risk_Off) if use_risk else False})
        ledger.update({'Date': target_date, 'Signal_Date': signal_date, 'Stage': 'Rebalance'})
        costs_rows.append(ledger)
        if LIQUIDATE_AT_END and i == len(TARGET_DATES)-1:
            liquidation = float((shares*price_after).sum())
            last_cost = liquidation*(BROKER_FEE_RATE+SELL_TAX_RATE)
            ending -= last_cost
            expected_liquidation = float((target_values*(1+predicted_returns/100)).sum())
            next_expected_net -= expected_liquidation*(BROKER_FEE_RATE+SELL_TAX_RATE)/pre_nav
            cash = ending
            shares[:] = 0
            costs_rows.append({'Date': target_date, 'Signal_Date': target_date, 'Stage': 'Liquidation',
                'Buy_Notional': 0.0, 'Sell_Notional': liquidation, 'Buy_Fee': 0.0,
                'Sell_Fee': liquidation*BROKER_FEE_RATE, 'Sell_Tax': liquidation*SELL_TAX_RATE,
                'Total_Cost': last_cost, 'Turnover': liquidation/(ending+last_cost)})
        if not np.isfinite(ending) or ending <= 0:
            raise ValueError('資金曲線無效。')
        nav_rows.append({'Date': target_date, 'NAV': ending, 'Cash': cash})
        direction_rows.append({'Date': target_date, 'Signal_Date': signal_date,
            'Reference_Actual_Return_Pct': reference_actual, 'Reference_Predicted_Return_Pct': reference_pred,
            'Reference_Direction_Correct': bool(direction([reference_actual])[0] == direction([reference_pred])[0]),
            'Dynamic_Gross_Actual_Return_Pct': dynamic_actual, 'Dynamic_Gross_Predicted_Return_Pct': dynamic_pred,
            'Dynamic_Gross_Direction_Correct': bool(direction([dynamic_actual])[0] == direction([dynamic_pred])[0]),
            'Net_Actual_Return_Pct': (ending/pre_nav-1)*100,
            'Net_Predicted_Return_Pct': next_expected_net*100,
            'Has_Exposure': bool(weights.sum() > 1e-10)})
    nav = pd.DataFrame(nav_rows).set_index('Date')
    nav['ROI_Pct'] = (nav.NAV/INITIAL_CAPITAL-1)*100
    nav['Drawdown_Pct'] = (nav.NAV/nav.NAV.cummax()-1)*100
    return {'nav': nav, 'weights': pd.DataFrame(weights_rows),
            'costs': pd.DataFrame(costs_rows), 'direction': pd.DataFrame(direction_rows),
            'terminal_shares': shares.copy(), 'terminal_cash': float(cash)}

BACKTEST = run_backtest()
display(BACKTEST['nav'].tail())
display(BACKTEST['costs'].sum(numeric_only=True).to_frame('Total'))

## 5．模型與績效指標的固定定義
年化 252 日、無風險利率 0、漲／跌／平容許值固定；金額與百分比欄位明確區分。
主要投資組合方向準確率用初始各 20% 固定股數的參考組合，與動態策略的獲利日比例、活躍日方向分開。未定義完整買賣交易勝率，不使用該名稱。

In [ ]:
def calculate_sharpe(values):
    a = np.asarray(values, dtype=float)
    returns = a[1:]/a[:-1]-1
    rf = (1+RISK_FREE_RATE)**(1/ANNUAL_TRADING_DAYS)-1
    std = returns.std(ddof=1) if len(returns) > 1 else 0.0
    return 0.0 if std < 1e-12 else float((returns.mean()-rf)/std*np.sqrt(ANNUAL_TRADING_DAYS))

def calculate_mdd(values):
    a = np.asarray(values, dtype=float)
    return float((a/np.maximum.accumulate(a)-1).min()*100)

def strategy_metrics(nav, name):
    series = pd.Series(nav, dtype=float)
    if series.isna().any() or not np.isfinite(series).all() or (series <= 0).any():
        raise ValueError('績效淨值不得含缺值或非正值。')
    return {'Strategy': name, 'ROI_Pct': (series.iloc[-1]/INITIAL_CAPITAL-1)*100,
            'Sharpe': calculate_sharpe(series), 'MDD_Pct': calculate_mdd(series),
            'Test_Days': len(series)-1, 'Test_Start': series.index[1].date().isoformat(),
            'Test_End': series.index[-1].date().isoformat()}

def model_metrics():
    rows = []
    for ticker, p in PREDICTIONS.items():
        rows.append({'Ticker': ticker, 'Test_Days': len(p),
            'Return_MAE_pp': mean_absolute_error(p.Actual_Return_Pct, p.Predicted_Return_Pct),
            'Return_RMSE_pp': np.sqrt(mean_squared_error(p.Actual_Return_Pct, p.Predicted_Return_Pct)),
            'Price_MAE': mean_absolute_error(p.Actual_Close, p.Predicted_Close),
            'Price_RMSE': np.sqrt(mean_squared_error(p.Actual_Close, p.Predicted_Close)),
            'Direction_Accuracy_Pct': p.Direction_Correct.mean()*100,
            'Always_Up_Accuracy_Pct': p.Actual_Direction.eq(1).mean()*100,
            'Zero_Return_MAE_pp': np.abs(p.Actual_Return_Pct).mean()})
    return pd.DataFrame(rows)

def plot_accuracy(table, correct_column, title):
    dates = pd.to_datetime(table.Date)
    correct = table[correct_column].astype(float)
    plt.figure(figsize=(11, 3))
    plt.plot(dates, correct.rolling(20).mean()*100, label='20-day rolling accuracy')
    plt.axhline(correct.mean()*100, color='firebrick', label=f'Overall: {correct.mean()*100:.2f}%')
    plt.axhline(50, color='gray', linestyle='--', label='50% reference only')
    plt.ylim(0, 100); plt.ylabel('Accuracy (%)'); plt.title(title)
    plt.grid(alpha=.25); plt.legend(); plt.tight_layout(); plt.show()

MODEL_METRICS = model_metrics()
display(MODEL_METRICS)

## 6．歷史回測、報酬與回撤
資金曲線包含初始本金，並扣除調倉成本及設定的期末賣出成本。圖與績效表使用同一份淨值。

In [ ]:
nav = BACKTEST['nav']
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(nav.index, nav.ROI_Pct, label='555 dynamic portfolio (net)', color='firebrick')
axes[0].axhline(0, color='gray', linewidth=.8); axes[0].set_ylabel('ROI (%)'); axes[0].legend()
axes[1].fill_between(nav.index, nav.Drawdown_Pct, 0, alpha=.3, color='firebrick')
axes[1].set(ylabel='Drawdown (%)', xlabel='Return realization date')
for ax in axes: ax.grid(alpha=.25)
fig.tight_layout(); plt.show()
display(pd.DataFrame([strategy_metrics(nav.NAV, '555 dynamic portfolio (net)')]))

## 7．五檔方向準確率與預測誤差
每檔各畫方向準確率及預測／實際報酬。超過 50% 不能直接宣稱顯著有效；一併查看固定猜上漲及零報酬基準。

In [ ]:
for ticker in TICKERS:
    p = PREDICTIONS[ticker]
    plot_accuracy(p, 'Direction_Correct', f'{ticker}: held-out direction accuracy')
    plt.figure(figsize=(11, 3))
    plt.plot(p.Date, p.Actual_Return_Pct, label='Actual return', alpha=.7)
    plt.plot(p.Date, p.Predicted_Return_Pct, label='Predicted return', alpha=.8)
    plt.title(ticker + ': next-day return'); plt.ylabel('Return (%)')
    plt.grid(alpha=.25); plt.legend(); plt.tight_layout(); plt.show()

## 8．投資組合方向準確率
主要指標為固定股數參考組合的預測方向，可與 444 比較。動態持股方向只在有持股時評估，另列樣本數。獲利日比例不叫「交易勝率」；本版未定義逐筆完整交易的開倉／平倉勝率。

In [ ]:
d = BACKTEST['direction']
active = d.Has_Exposure
PORTFOLIO_METRICS = pd.DataFrame([{
    'Reference_Direction_Accuracy_Pct': d.Reference_Direction_Correct.mean()*100,
    'Always_Up_Reference_Accuracy_Pct': (direction(d.Reference_Actual_Return_Pct) == 1).mean()*100,
    'Dynamic_Active_Direction_Accuracy_Pct': d.loc[active, 'Dynamic_Gross_Direction_Correct'].mean()*100,
    'Active_Days': int(active.sum()), 'All_Cash_Days': int((~active).sum()),
    'Profitable_Net_Day_Pct': (d.Net_Actual_Return_Pct > 0).mean()*100}])
display(PORTFOLIO_METRICS)
plot_accuracy(d, 'Reference_Direction_Correct', '555: fixed-reference portfolio direction accuracy')

## 9．555、大盤 Sharpe／MDD 與日報酬相關係數
大盤為 ^TWII 理論價格指數，未扣券商手續費，不是可直接成交的 ETF。相關係數使用相同日期的日報酬，而不是累積淨值；零變異時無法定義相關係數。

In [ ]:
market = INITIAL_CAPITAL*PRICE_PANEL.loc[nav.index, MARKET_TICKER]/PRICE_PANEL.loc[nav.index[0], MARKET_TICKER]
market.name = 'Market_Index'
market_metrics = strategy_metrics(market, 'TWII theoretical price index')
new_metrics = strategy_metrics(nav.NAV, '555 multivariate + dynamic risk (net)')
new_metrics['Direction_Accuracy_Pct'] = d.Reference_Direction_Correct.mean()*100
STRATEGY_METRICS = pd.DataFrame([new_metrics])
MARKET_METRICS = pd.DataFrame([market_metrics])
display(STRATEGY_METRICS)
display(MARKET_METRICS)
daily = pd.DataFrame({'New_Return': nav.NAV.pct_change(fill_method=None),
                      'Market_Return': market.pct_change(fill_method=None)}).dropna()
correlation = (daily.New_Return.corr(daily.Market_Return)
               if daily.New_Return.std() > 1e-12 and daily.Market_Return.std() > 1e-12 else np.nan)
display(pd.DataFrame([{'Daily_Return_Pearson_Correlation': correlation}]))
plt.figure(figsize=(5, 4))
plt.scatter(daily.Market_Return*100, daily.New_Return*100, alpha=.55)
plt.xlabel('TWII daily return (%)'); plt.ylabel('555 daily net return (%)')
plt.title(f'Daily return correlation: {correlation:.3f}'); plt.grid(alpha=.25); plt.tight_layout(); plt.show()

## 10．每日股票／現金權重與市場風控
保留 50% 現金是持股上限規則，不代表每天必須投滿剩餘 50%。圖中曝險及現金為調倉後的目標比例。

In [ ]:
weights = BACKTEST['weights'].pivot(index='Date', columns='Ticker', values='Weight')[TICKERS]
weights['Cash'] = 1-weights.sum(axis=1)
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].stackplot(weights.index, *[weights[c].to_numpy() for c in weights], labels=weights.columns)
axes[0].set_ylim(0, 1); axes[0].set_ylabel('Target weight'); axes[0].legend(loc='upper left', ncol=3)
signals = PREDICTIONS[TICKERS[0]]
axes[1].plot(signals.Date, signals.Market_Vol_20d, label='20-day annualized volatility (%)')
axes[1].plot(signals.Date, signals.Risk_Threshold_Pct, label='Past mean + 1 std', linestyle='--')
axes[1].set(ylabel='Volatility (%)', xlabel='Return realization date'); axes[1].legend()
for ax in axes: ax.grid(alpha=.25)
fig.tight_layout(); plt.show()
display(pd.DataFrame([{'Average_Cash_Weight_Pct': weights.Cash.mean()*100,
                      'Risk_Off_Days': int(signals.Risk_Off.sum()),
                      'Total_Transaction_Cost': BACKTEST['costs'].Total_Cost.sum(),
                      'Total_Turnover': BACKTEST['costs'].Turnover.sum()}]))

## 11．跨日檢討：只讀本版以前保存的建議
重訓後的模型不能倒回去改寫昨天的預測。檢討使用本版真正保存的預測，按 Signal_Date／Target_Date 對齊實際價格。
同一交易日若重跑多次，選目標日開盤前最後一份 daily 建議。歷史重播建議不放入正式每日紀錄；目標日尚未發生者標示 Pending。組合回報是推薦權重的理論毛回報，不能當作實際帳戶淨收益。

In [ ]:
REVIEW_COLUMNS = ['Created_At','Run_ID','Run_Mode','Signal_Date','Target_Date','Ticker',
    'Predicted_Return_Pct','Target_Weight','Cash_Weight','Actual_Return_Pct','Return_Error_pp',
    'Direction_Correct','Review_Status']
PORTFOLIO_REVIEW_COLUMNS = ['Target_Date','Run_ID','Predicted_Gross_Return_Pct','Actual_Gross_Return_Pct',
    'Gross_Direction_Correct','Has_Exposure','Cash_Weight','Status']

def review_past_recommendations():
    history_dir = Path(os.environ.get('LSTM_RECOMMENDATION_HISTORY_DIR', str(OUTPUT_ROOT/'recommendation_history')))
    if not history_dir.exists():
        return pd.DataFrame(columns=REVIEW_COLUMNS), pd.DataFrame(columns=PORTFOLIO_REVIEW_COLUMNS)
    parts = []
    for path in sorted(history_dir.glob('*.csv')):
        part = pd.read_csv(path)
        required = {'Created_At','Run_ID','Run_Mode','Signal_Date','Target_Date','Ticker',
                    'Predicted_Return_Pct','Target_Weight','Cash_Weight'}
        if not required.issubset(part):
            raise ValueError(f'以前的建議格式不完整：{path.name}')
        part['Created_At'] = pd.to_datetime(part.Created_At, utc=True)
        part['Signal_Date'] = pd.to_datetime(part.Signal_Date).dt.normalize()
        part['Target_Date'] = pd.to_datetime(part.Target_Date).dt.normalize()
        if part[['Created_At','Signal_Date','Target_Date']].isna().any().any():
            raise ValueError('歷史建議含缺失時間。')
        numbers = part[['Predicted_Return_Pct','Target_Weight','Cash_Weight']].to_numpy(dtype=float)
        if not np.isfinite(numbers).all():
            raise ValueError('歷史建議含非有限預測或權重。')
        cutoff = ((part.Target_Date+pd.Timedelta(hours=9)).dt.tz_localize('Asia/Taipei').dt.tz_convert('UTC'))
        now_utc = pd.Timestamp(CREATED_AT).tz_convert('UTC')
        part = part.loc[(part.Run_Mode == 'daily') & (part.Created_At < cutoff)
                        & (part.Created_At <= now_utc)].copy()
        if not part.empty:
            parts.append(part)
    if not parts:
        return pd.DataFrame(columns=REVIEW_COLUMNS), pd.DataFrame(columns=PORTFOLIO_REVIEW_COLUMNS)
    candidates = pd.concat(parts, ignore_index=True)
    selected_runs = (candidates[['Target_Date','Run_ID','Created_At']].drop_duplicates()
                     .sort_values('Created_At').drop_duplicates('Target_Date', keep='last'))
    candidates = candidates.merge(selected_runs[['Target_Date','Run_ID']], on=['Target_Date','Run_ID'])
    if candidates.duplicated(['Target_Date','Ticker']).any():
        raise ValueError('歷史建議存在重複股票／日期。')
    reviewed, portfolio_rows = [], []
    for target_date, group in candidates.groupby('Target_Date'):
        if set(group.Ticker) != set(TICKERS) or len(group) != len(TICKERS):
            raise ValueError('歷史建議不是完整五檔，不能評估組合。')
        group = group.set_index('Ticker').loc[TICKERS].reset_index()
        if group.Signal_Date.nunique() != 1:
            raise ValueError('歷史組合訊號日不一致。')
        signal_date = group.Signal_Date.iloc[0]
        if signal_date >= target_date or trading_sessions(signal_date+pd.Timedelta(days=1), target_date).size != 1:
            raise ValueError('歷史建議不是下一交易日預測。')
        if (group.Target_Weight < 0).any() or group.Target_Weight.sum() > 1+1e-8:
            raise ValueError('歷史建議權重無效。')
        if not np.allclose(group.Cash_Weight, 1-group.Target_Weight.sum(), atol=1e-8):
            raise ValueError('歷史建議現金比例與股票權重不一致。')
        status = 'Pending' if target_date > RESOLVED_AS_OF else 'Price_Unavailable'
        actuals = []
        for _, r in group.iterrows():
            record = r.to_dict()
            if target_date in PRICE_PANEL.index and signal_date in PRICE_PANEL.index:
                actual = float((PRICE_PANEL.loc[target_date,r.Ticker]/PRICE_PANEL.loc[signal_date,r.Ticker]-1)*100)
                record.update({'Actual_Return_Pct': actual,
                               'Return_Error_pp': float(r.Predicted_Return_Pct-actual),
                               'Direction_Correct': bool(direction([r.Predicted_Return_Pct])[0] == direction([actual])[0]),
                               'Review_Status': 'Reviewed'})
                actuals.append(actual)
            else:
                record.update({'Actual_Return_Pct': np.nan, 'Return_Error_pp': np.nan,
                               'Direction_Correct': np.nan, 'Review_Status': status})
            reviewed.append(record)
        if len(actuals) == len(TICKERS):
            w = group.Target_Weight.to_numpy(dtype=float)
            pred = float(w@group.Predicted_Return_Pct.to_numpy(dtype=float))
            actual = float(w@np.array(actuals))
            portfolio_rows.append({'Target_Date': target_date, 'Run_ID': group.Run_ID.iloc[0],
                'Predicted_Gross_Return_Pct': pred, 'Actual_Gross_Return_Pct': actual,
                'Gross_Direction_Correct': bool(direction([pred])[0] == direction([actual])[0]) if w.sum() > 0 else np.nan,
                'Has_Exposure': bool(w.sum() > 0), 'Cash_Weight': 1-w.sum(),
                'Status': 'Hypothetical recommendation, gross; not an account PnL'})
    return pd.DataFrame(reviewed), pd.DataFrame(portfolio_rows,columns=PORTFOLIO_REVIEW_COLUMNS)

DAILY_REVIEW, DAILY_PORTFOLIO_REVIEW = review_past_recommendations()
if DAILY_REVIEW.empty:
    print('首次執行或尚無可檢討的每日建議。從本次保存開始累積，不回填歷史預測。')
else:
    display(DAILY_REVIEW.tail(10))
    complete = DAILY_REVIEW.loc[DAILY_REVIEW.Review_Status == 'Reviewed']
    if not complete.empty:
        display(pd.DataFrame([{'Reviewed_Stock_Days': len(complete),
            'Direction_Accuracy_Pct': complete.Direction_Correct.astype(float).mean()*100,
            'Return_MAE_pp': complete.Return_Error_pp.abs().mean()}]))
        display(DAILY_PORTFOLIO_REVIEW.tail())

## 12．訓練損失與可選策略規則消融
開關只關閉配置規則，LSTM 仍保留六個輸入；不能把這一節當成移除新聞特徵的完整模型實驗。

In [ ]:
for ticker, history in HISTORIES.items():
    plt.figure(figsize=(7, 3))
    for key in ['loss', 'val_loss']:
        plt.plot(np.arange(1, len(history[key])+1), history[key], label=key)
    plt.title(ticker + ': training and validation MSE'); plt.xlabel('Epoch')
    plt.grid(alpha=.25); plt.legend(); plt.tight_layout(); plt.show()
ABLATION_TABLE = pd.DataFrame()
if RUN_ABLATIONS:
    ablation_rows = []
    for label, sentiment_on, risk_on in [
        ('Prediction-only allocation', False, False),
        ('Sentiment allocation, no risk filter', True, False),
        ('Full dynamic allocation', True, True)]:
        result = run_backtest(sentiment_on, risk_on)
        row = strategy_metrics(result['nav'].NAV, label)
        row['Total_Cost'] = result['costs'].Total_Cost.sum()
        ablation_rows.append(row)
    ABLATION_TABLE = pd.DataFrame(ablation_rows)
    display(ABLATION_TABLE)

## 13．下一交易日預測與投資權重
評估模型與部署模型分開保存。部署模型以截至截止日所有已知答案重訓，epoch 數由歷史驗證集選定；不拿它回頭計算歷史勝率。
本版每次完整執行都會訓練評估模型，並在啟用時重訓部署模型；檢討昨日建議在重訓部署模型前完成。每日紀錄保存原始預測，不做交易下單。

In [ ]:
def next_trading_date(last_date):
    start = pd.Timestamp(last_date)+pd.Timedelta(days=1)
    sessions = trading_sessions(start, start+pd.Timedelta(days=60))
    if len(sessions) == 0:
        raise ValueError('無法取得下一交易日。')
    return sessions[0]

def next_day_recommendation():
    latest = PRICE_PANEL.index[-1]
    predictions, sentiments = [], []
    # 先檢查全部股票，再重訓，避免資料過期卻耗時訓練。
    for ticker in TICKERS:
        window = FEATURE_FRAMES[ticker][FEATURES].iloc[-TIME_STEP:]
        if len(window) != TIME_STEP or not np.isfinite(window.to_numpy()).all():
            READINESS_NOTES.append(f'{ticker} 最新十日特徵不完整，下一日建議暫緩。')
            return pd.DataFrame()
    if not np.isfinite(MARKET_RISK.Risk_Threshold_Pct.iloc[-1]):
        READINESS_NOTES.append('最新風險門檻無效，下一日建議暫緩。')
        return pd.DataFrame()
    for ticker in TICKERS:
        frame = FEATURE_FRAMES[ticker]
        data = MODELS[ticker]['data']
        all_rows = np.unique(np.concatenate([np.arange(e-TIME_STEP+1, e+1) for e in data['ends']]))
        scaler = MinMaxScaler().fit(frame[FEATURES].iloc[all_rows].to_numpy())
        X = np.stack([scaler.transform(frame[FEATURES].iloc[e-TIME_STEP+1:e+1].to_numpy())
                      for e in data['ends']])
        model = model_factory()
        best_epoch = MODELS[ticker]['best_epoch']
        model.fit(X, data['y_all'], epochs=best_epoch, batch_size=BATCH_SIZE, shuffle=False, verbose=0)
        latest_window = scaler.transform(frame[FEATURES].iloc[-TIME_STEP:].to_numpy())
        pred = float(model.predict(latest_window[None, ...], verbose=0).ravel()[0])
        if not np.isfinite(pred) or pred <= -100:
            raise ValueError('部署預測无效。')
        predictions.append(pred)
        sentiments.append(float(frame.Daily_Sentiment.iloc[-1]))
        bundle = {'model': model, 'scaler': scaler, 'best_epoch': best_epoch}
        DEPLOYMENT_MODELS[ticker] = bundle
        save_bundle(bundle, ticker, 'deployment')
    risk_off = bool(MARKET_RISK.Risk_Off.iloc[-1])
    target_weights = allocate(predictions, sentiments, risk_off)
    target_date = next_trading_date(latest)
    table = pd.DataFrame({'Run_ID': RUN_ID, 'Signal_Date': latest,
        'Target_Date': target_date, 'Ticker': TICKERS,
        'Created_At': CREATED_AT.isoformat(), 'Run_Mode': RUN_MODE,
        'Predicted_Return_Pct': predictions, 'Daily_Sentiment': sentiments,
        'Target_Weight': target_weights, 'Cash_Weight': 1-target_weights.sum(),
        'Market_Vol_20d': MARKET_RISK.Market_Vol_20d.iloc[-1],
        'Risk_Threshold_Pct': MARKET_RISK.Risk_Threshold_Pct.iloc[-1],
        'Risk_Off': risk_off, 'News_Time_Audit_Confirmed': NEWS_TIME_AUDIT_CONFIRMED,
        'Status': 'Research recommendation; no order submitted'})
    return table

NEXT_DAY = next_day_recommendation() if RUN_DEPLOYMENT_REFIT else pd.DataFrame()
if not NEXT_DAY.empty:
    display(NEXT_DAY)
else:
    print('尚未產出下一交易日建議；請查看待確認項目或啟用部署重訓。')

## 可選．未來 30 日情境與時光機
設定 `RUN_SCENARIO_FORECAST=True` 才執行。需要已訓練部署模型。第一步用最新真實特徵；後續情緒固定為 SCENARIO_SENTIMENT，大盤波動率及門檻固定為最後已知值，技術指標用推估價格更新。
這是明示假設的敏感度情境，不能與每日實測績效混在一起。資金由历史回測期末状态延伸，遵循同樣調倉成本；未來區段不在期末強制卖出。

In [ ]:
SCENARIO_TABLE = pd.DataFrame()
if RUN_SCENARIO_FORECAST:
    if len(DEPLOYMENT_MODELS) != len(TICKERS):
        raise ValueError('情境外推需要先完成全部部署模型。')
    if not -1 <= SCENARIO_SENTIMENT <= 1:
        raise ValueError('情境情緒須在 [-1,1]。')
    start = PRICE_PANEL.index[-1]+pd.Timedelta(days=1)
    dates = trading_sessions(start,start+pd.Timedelta(days=SCENARIO_DAYS*4+60))[:SCENARIO_DAYS]
    if len(dates) != SCENARIO_DAYS:
        raise ValueError('情境交易日期不足。')
    series = {t:PRICE_PANEL[t].copy() for t in TICKERS}
    windows = {t:FEATURE_FRAMES[t][FEATURES].iloc[-TIME_STEP:].copy() for t in TICKERS}
    shares = np.zeros(len(TICKERS))
    cash = float(BACKTEST['nav'].NAV.iloc[-1])
    if not LIQUIDATE_AT_END:
        last_weights = BACKTEST['weights'].loc[BACKTEST['weights'].Date == TARGET_DATES[-1]].set_index('Ticker').loc[TICKERS,'Weight'].to_numpy()
        # Exact terminal stock/cash values are saved by the backtest for this continuation.
        shares = BACKTEST['terminal_shares'].copy()
        cash = BACKTEST['terminal_cash']
    rows = []
    for date in dates:
        preds, sentiments = [], []
        before = np.array([series[t].iloc[-1] for t in TICKERS])
        for t in TICKERS:
            bundle = DEPLOYMENT_MODELS[t]
            window = bundle['scaler'].transform(windows[t].to_numpy())
            prediction = float(bundle['model'].predict(window[None,...],verbose=0).ravel()[0])
            if not np.isfinite(prediction) or prediction <= -100:
                raise ValueError('情境模型產生無效價格，已停止，不能截斷掩蓋。')
            preds.append(prediction)
            sentiments.append(float(windows[t].Daily_Sentiment.iloc[-1]))
        w = allocate(preds,sentiments,bool(MARKET_RISK.Risk_Off.iloc[-1]))
        target_values,cash,ledger = rebalance(shares*before,cash,w)
        shares = target_values/before
        after = before*(1+np.array(preds)/100)
        value = float(shares@after+cash)
        record = {'Date':date,'Scenario_NAV':value,'Scenario_Cash':cash,'Total_Cost':ledger['Total_Cost'],
                  'Segment':'Scenario','Assumed_Future_Sentiment':SCENARIO_SENTIMENT,
                  'Assumed_Market_Vol_20d':MARKET_RISK.Market_Vol_20d.iloc[-1]}
        for j,t in enumerate(TICKERS):
            series[t] = pd.concat([series[t],pd.Series([after[j]],index=[date])])
            updated = technical_features(series[t]).iloc[-1].to_dict()
            updated['Market_Vol_20d'] = MARKET_RISK.Market_Vol_20d.iloc[-1]
            updated['Daily_Sentiment'] = SCENARIO_SENTIMENT
            windows[t] = pd.concat([windows[t].iloc[1:],pd.DataFrame([updated],index=[date])[FEATURES]])
            record[t+'_Scenario_Close'] = after[j]
        rows.append(record)
    SCENARIO_TABLE = pd.DataFrame(rows)
    SCENARIO_TABLE.to_csv(OUTPUT_DIR/'future_scenario.csv',index=False,encoding='utf-8-sig')
    plt.figure(figsize=(11,4))
    connection_dates = pd.DatetimeIndex([PRICE_PANEL.index[-1]]).append(pd.DatetimeIndex(SCENARIO_TABLE.Date))
    connection_nav = np.r_[BACKTEST['nav'].NAV.iloc[-1],SCENARIO_TABLE.Scenario_NAV]
    plt.plot(connection_dates,(connection_nav/connection_nav[0]-1)*100,linestyle='--',label='30-day assumed scenario')
    plt.title('Future scenario: sentiment assumed, market volatility frozen')
    plt.ylabel('Scenario return from latest NAV (%)');plt.grid(alpha=.25);plt.legend();plt.tight_layout();plt.show()
    plt.figure(figsize=(11,4))
    history = BACKTEST['nav'].NAV.iloc[-100:]
    plt.plot(history.index,(history/INITIAL_CAPITAL-1)*100,label='Observed backtest, net')
    connection = pd.concat([history.iloc[-1:],pd.Series(SCENARIO_TABLE.Scenario_NAV.to_numpy(),index=SCENARIO_TABLE.Date)])
    plt.plot(connection.index,(connection/INITIAL_CAPITAL-1)*100,linestyle='--',label='Assumed future scenario')
    plt.axvline(history.index[-1],color='gray',linestyle=':')
    plt.title('Time machine: observed history and separate assumed scenario')
    plt.ylabel('ROI (%)');plt.grid(alpha=.25);plt.legend();plt.tight_layout();plt.show()
    display(SCENARIO_TABLE.tail())
else:
    print('多日情境未啟用。下一交易日建議與真實歷史指標仍正常输出。')

## 14．本版標準化匯出、不可覆盖日誌與每日快照
每次執行保存至 `lstm_multivariate_outputs/runs/<Run_ID>/`，不覆盖舊模型、預測或指標。只有整次匯出成功後才更新 `latest.json`，前端日後可由此找到最後成功結果。
輸出共同欄位、日淨值、實際報酬、資料截止日、測試／驗證／訓練日期及 SHA-256。每日模式的下一日建議另存至本版 `recommendation_history/`；歷史重播不加入正式每日紀錄。
執行失敗不會更新 latest 指標；GitHub Actions 還需另外設定 runner、排程、資料端與失敗通知。

In [ ]:
def date_list_hash(dates):
    text = '\n'.join(pd.DatetimeIndex(dates).strftime('%Y-%m-%d'))
    return hashlib.sha256(text.encode()).hexdigest()

def make_protocol():
    return {'protocol_version': PROTOCOL_VERSION, 'strategy_id': STRATEGY_ID,
        'timezone': 'Asia/Taipei', 'data_ready_time': DATA_READY_TIME,
        'as_of_date': RESOLVED_AS_OF.date().isoformat(),
        'end_exclusive': END_DATE_EXCLUSIVE.date().isoformat(),
        'requested_history_start': START_DATE, 'tickers': TICKERS, 'market_ticker': MARKET_TICKER,
        'price_auto_adjust': AUTO_ADJUST, 'dividends_reinvested': False,
        'fractional_shares': True, 'cash_return': 0.0,
        'time_step': TIME_STEP, 'test_days': TEST_DAYS, 'validation_days': VALIDATION_DAYS,
        'test_target_dates': TARGET_DATES.strftime('%Y-%m-%d').tolist(),
        'validation_target_dates': VALIDATION_TARGET_DATES.strftime('%Y-%m-%d').tolist(),
        'test_target_dates_sha256': date_list_hash(TARGET_DATES),
        'validation_target_dates_sha256': date_list_hash(VALIDATION_TARGET_DATES),
        'initial_capital': INITIAL_CAPITAL, 'broker_fee_rate': BROKER_FEE_RATE,
        'sell_tax_rate': SELL_TAX_RATE, 'liquidate_at_end': LIQUIDATE_AT_END,
        'annual_trading_days': ANNUAL_TRADING_DAYS, 'risk_free_rate': RISK_FREE_RATE,
        'direction_zero_tolerance_pct': DIRECTION_ZERO_TOL_PCT,
        'direction_definition': 'sign(next close / signal close - 1), flat within tolerance',
        'portfolio_direction_definition': 'initial equal-weight fixed-share reference; excludes costs',
        'roi_definition': '(last net NAV / initial capital - 1) * 100',
        'mdd_definition': 'min(NAV / running peak - 1) * 100; includes initial NAV; negative percent',
        'sharpe_definition': 'sqrt(252) * mean(daily return - daily rf) / sample std(daily return)',
        'execution_assumption': 'idealized signal-day close to next-day close; no slippage',
        'seed': SEED, 'epochs_max': EPOCHS, 'patience': PATIENCE, 'batch_size': BATCH_SIZE,
        'train_validation_protocol': 'fixed chronological target dates; train-only scaler; early stopping',
        'model_target': 'next-trading-day return in percentage points',
        'price_snapshot_sha256': INPUT_HASHES['price_snapshot'],
        'price_panel_sha256': hashlib.sha256(PRICE_PANEL.to_csv(float_format='%.10f').encode()).hexdigest(),
        'actual_split_audit': SPLIT_AUDIT,
        'feature_columns': FEATURES, 'news_time_audit_confirmed': NEWS_TIME_AUDIT_CONFIRMED,
        'feature_availability_note': 'actual training coverage must be checked before later comparison'}

def export_results():
    metrics = STRATEGY_METRICS.copy()
    metrics.insert(0, 'Strategy_ID', STRATEGY_ID)
    metrics.insert(1, 'Run_ID', RUN_ID)
    metrics['As_Of_Date'] = RESOLVED_AS_OF.date().isoformat()
    metrics['Protocol_Version'] = PROTOCOL_VERSION
    tables = {'strategy_metrics.csv': metrics, 'market_metrics.csv': MARKET_METRICS,
        'model_metrics.csv': MODEL_METRICS,
        'test_predictions.csv': pd.concat(PREDICTIONS.values(), ignore_index=True),
        'split_audit.csv': pd.DataFrame(SPLIT_AUDIT),
        'portfolio_direction.csv': BACKTEST['direction'], 'portfolio_metrics.csv': PORTFOLIO_METRICS,
        'daily_weights.csv': BACKTEST['weights'], 'transaction_costs.csv': BACKTEST['costs'],
        'daily_prediction_review.csv': DAILY_REVIEW,
        'daily_portfolio_review.csv': DAILY_PORTFOLIO_REVIEW,
        'daily_return_correlation.csv': pd.DataFrame([{'Daily_Return_Pearson_Correlation':correlation}]),
        'risk_summary.csv': pd.DataFrame([{'Average_Cash_Weight_Pct':weights.Cash.mean()*100,
            'Risk_Off_Days':int(PREDICTIONS[TICKERS[0]].Risk_Off.sum()),
            'Total_Transaction_Cost':BACKTEST['costs'].Total_Cost.sum(),
            'Total_Turnover':BACKTEST['costs'].Turnover.sum()}]),
        'figure_index.csv': pd.DataFrame(FIGURE_INDEX)}
    if not ABLATION_TABLE.empty:
        tables['allocation_rule_ablation.csv'] = ABLATION_TABLE
    for filename, table in tables.items():
        table.to_csv(OUTPUT_DIR/filename, index=False, encoding='utf-8-sig')
    nav_export = BACKTEST['nav'].copy()
    nav_export['Daily_Return_Pct'] = nav_export.NAV.pct_change(fill_method=None)*100
    nav_export['Market_Index'] = market
    nav_export.to_csv(OUTPUT_DIR/'backtest_nav.csv', encoding='utf-8-sig')
    for ticker, history in HISTORIES.items():
        training = pd.DataFrame(history)
        training.index = np.arange(1,len(training)+1)
        training.rename_axis('Epoch').to_csv(OUTPUT_DIR/f'training_history_{ticker.replace(".","_")}.csv', encoding='utf-8-sig')
    if not NEXT_DAY.empty:
        NEXT_DAY.to_csv(OUTPUT_DIR/'next_day_recommendation.csv', index=False, encoding='utf-8-sig')
    protocol = make_protocol()
    (OUTPUT_DIR/'evaluation_protocol.json').write_text(json.dumps(protocol,ensure_ascii=False,indent=2),encoding='utf-8')
    config = {'run_id':RUN_ID, 'created_at':CREATED_AT.isoformat(), 'run_mode':RUN_MODE,
        'protocol':protocol, 'input_sha256':INPUT_HASHES,
        'sentiment_bull':SENTIMENT_BULL, 'sentiment_bear':SENTIMENT_BEAR,
        'max_weight':MAX_WEIGHT_LIMIT, 'risk_exposure':RISK_EXPOSURE,
        'risk_threshold':'prior expanding annualized volatility mean + sample std; min 60',
        'news_time_audit_confirmed':NEWS_TIME_AUDIT_CONFIRMED,
        'pending_items':sorted(set(READINESS_NOTES)),
        'versions':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,
                    'scikit_learn':sklearn.__version__,'tensorflow':getattr(tf,'__version__',None)}}
    (OUTPUT_DIR/'run_config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
    report = '# 555 每日工作日誌\n\n'
    report += f'Run_ID: {RUN_ID}\n\n資料截至：{RESOLVED_AS_OF.date()}；模式：{RUN_MODE}\n\n'
    report += '## 新版實测績效\n\n```text\n'+metrics.to_string(index=False)+'\n```\n\n'
    report += '## 截止日個股報酬、預測與情緒\n\n```text\n'
    today_results = pd.concat([p.tail(1) for p in PREDICTIONS.values()],ignore_index=True)
    report += today_results[['Date','Ticker','Actual_Return_Pct','Predicted_Return_Pct','Daily_Sentiment']].to_string(index=False)+'\n```\n\n'
    report += '情緒欄為該次預測訊號日的情緒，不能誤標為報酬實現日新聞。\n\n'
    report += '## 以前保存建議的跨日檢討\n\n```text\n'+DAILY_REVIEW.tail(10).to_string(index=False)+'\n```\n\n'
    report += '## 最新情緒與下一交易日權重\n\n```text\n'+NEXT_DAY.to_string(index=False)+'\n```\n\n'
    report += '## 待確認事項\n\n'+'\n'.join('- '+x for x in sorted(set(READINESS_NOTES)))
    report += '\n\n每日重訓已在本次執行流程完成；GitHub 排程尚待外部設定。推薦組合回報為理論毛回報，非實際帳戶淨收益。\n'
    (OUTPUT_DIR/'work_log.md').write_text(report,encoding='utf-8')
    output_hashes = {str(p.relative_to(OUTPUT_DIR)):sha256_file(p)
                     for p in sorted(OUTPUT_DIR.rglob('*')) if p.is_file()}
    manifest = {'status':'completed','run_id':RUN_ID,'as_of_date':RESOLVED_AS_OF.date().isoformat(),
                'created_at':CREATED_AT.isoformat(),'run_mode':RUN_MODE,
                'output_directory':str(OUTPUT_DIR.relative_to(OUTPUT_ROOT)),
                'protocol_version':PROTOCOL_VERSION,'output_sha256':output_hashes}
    (OUTPUT_DIR/'run_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
    print('完整執行結果已匯出：',OUTPUT_DIR.resolve())
    return config, manifest

RUN_CONFIG, RUN_MANIFEST = export_results()
import shutil
archive = shutil.make_archive(str(OUTPUT_DIR), 'zip', root_dir=OUTPUT_DIR)
if RUN_MODE == 'daily' and not NEXT_DAY.empty:
    history_dir = Path(os.environ.get('LSTM_RECOMMENDATION_HISTORY_DIR', str(OUTPUT_ROOT/'recommendation_history')))
    history_dir.mkdir(parents=True, exist_ok=True)
    history_path = history_dir/(RUN_ID+'.csv')
    if history_path.exists():
        raise FileExistsError('不可覆盖既有每日建議。')
    temporary_history = history_dir/(RUN_ID+'.tmp')
    NEXT_DAY.to_csv(temporary_history,index=False,encoding='utf-8-sig')
    temporary_history.replace(history_path)
# 全部寫出／打包成功才更新；歷史重播不移動 daily latest。
pointer_name = 'latest.json' if RUN_MODE == 'daily' else 'latest_replay.json'
temporary_pointer = OUTPUT_ROOT/(pointer_name+'.'+RUN_ID+'.tmp')
temporary_pointer.write_text(json.dumps(RUN_MANIFEST,ensure_ascii=False,indent=2),encoding='utf-8')
temporary_pointer.replace(OUTPUT_ROOT/pointer_name)
print('本次壓縮檔：',archive)
try:
    from google.colab import files
except ImportError:
    print('本機執行：請開啟本次 ZIP；保留 OUTPUT_ROOT 才能累積跨日檢討。')
else:
    files.download(archive)

## 獨立運作與之後同步的交接規則

1. **555 已提供**：獨立價格讀取、情緒驗證、技術／市場風險、時序切分、全部模型與指標、動態成本、股票／現金權重、歷史預測與部署模型分離、自己以前保存建議的跨日檢討、每日不可覆盖快照及工作日誌。
2. **每日執行順序**：上游先更新新聞情緒 → 執行此 Notebook → 讀 work_log、strategy_metrics、next_day_recommendation。GitHub 排程需另設定 repository 與 runner；本版不直接上传 repo、發訊息或下單。
3. **需持久保存**：`runs/`、`recommendation_history/`、`latest.json`。Colab runtime 會重置；每日運作需在持久磁碟或 GitHub 產物下載／保存後再跑，否则无法累積跨日檢討。
4. **之後接收端使用**：evaluation_protocol.json、strategy_metrics.csv、test_predictions.csv、backtest_nav.csv、split_audit.csv、market_metrics.csv 及資料雜湊。新舊比較只能在日期、價格、成本、指標定義等一致時進行；訓練範圍也須核對。這份程式不執行新舊比較。
5. **尚待研究驗證**：真實新聞來源與時間映射、相同預測目標／交易規則的特徵實驗、策略規則消融、多 seed／多期間穩健性、close-to-close 執行假設及實際成交限制。
6. **未來 30 日與時光機**：多變量外生資訊未知。可選情境模块明示未來情緒假設及固定市場波動，輸出是 Scenario，不列入歷史 ROI／勝率／Sharpe，不假稱真實未来預測。下一交易日建議使用最新已知特徵，另保存。
7. **前端**：依組長指示暫緩，等各自每日產物穩定再接 Streamlit。結果及報告不預設新版必定勝出。